# FINS5557 Week 9 — WealthTech Financial Advisory Platform

**Cross-reference:** Week 9 Seminar Notes, Section 9.4 (Case Study)

This notebook implements an AI-augmented financial advisory platform used by licensed financial advisers to serve private clients. The platform operationalises the *Humanised Generative AI* model proposed by Lo and Ross [2024]: the LLM drafts the Statement of Advice; the adviser reviews and decides; the audit trail proves that the Best Interests Duty (Corporations Act s.961B) was discharged.

## DDF Pipeline Mapping (Table 9.3)

| DDF Station | This Notebook |
|---|---|
| **ETL (Station 1)** | Synthetic client profiles (CRM data) + product database (managed funds, ETFs, super, term deposits) |
| **Feature Engineering (Station 2)** | Client risk score CRS (Eq 9.6) · Product suitability score PSS (Eq 9.7) · Goal-product compatibility |
| **Model Design (Station 3)** | Recommendation ranking · Best Interests Duty compliance gate · Portfolio allocation (IPS targets) |
| **Implementation (Station 4)** | Gemini/mock SOA draft generation · Adviser audit trail · Streamlit adviser workstation |

## Setup Instructions (Google Colab)

1. Get a **free** Gemini API key at `https://aistudio.google.com/apikey` — no credit card required.
2. In Colab: key icon (Secrets) → `+ Add new secret` → Name: `GOOGLE_API_KEY`, Value: paste key, toggle **Notebook access** ON.
3. Re-run Cell 2 — the notebook auto-detects the secret and enables live SOA draft generation via Gemini 1.5 Flash.
4. Without a key, a complete mock SOA is generated automatically — all pipeline stages produce representative adviser-ready output.


In [ ]:
# ── Cell 2: Setup — API key detection + library imports ───────────────────
import subprocess, sys

def _install(pkg):
    subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'], check=True)

try:
    import google.generativeai as genai
except ImportError:
    _install('google-generativeai')
    import google.generativeai as genai

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings, datetime, hashlib

warnings.filterwarnings('ignore')
np.random.seed(42)

# ── Gemini API key (Colab Secrets → env var → mock fallback) ──────────────
def _load_api_key():
    try:
        from google.colab import userdata
        key = userdata.get('GOOGLE_API_KEY')
        if key:
            print('API key loaded from Colab Secrets.')
            return key
    except Exception:
        pass
    import os
    return os.getenv('GOOGLE_API_KEY')

GEMINI_API_KEY = _load_api_key()
USE_LIVE_API   = GEMINI_API_KEY is not None
GEMINI_MODEL   = 'gemini-1.5-flash'

if USE_LIVE_API:
    genai.configure(api_key=GEMINI_API_KEY)
    print(f'Live Gemini API enabled: {GEMINI_MODEL}')
else:
    print('No API key — running mock simulation.')
    print('Add GOOGLE_API_KEY to Colab Secrets to enable live SOA generation.')


## Station 1: ETL — Client Profiles and Product Database

The platform ingests two primary data sources: a client master database (drawn from the firm's CRM) and a product shelf maintained by the investment committee. The product shelf includes managed funds, ETFs, superannuation investment options, term deposits, and risk insurance — each with a standardised set of features used by the recommendation engine.


In [ ]:
# ── Station 1: ETL — Client profiles + product database ──────────────────

# ── Client master data (CRM extract — anonymised for demo) ────────────────
CLIENTS = [
    {'client_id': 'CLI_001', 'name': 'Sarah Chen',      'age': 45,
     'investable_aud': 850_000,  'income_aud': 180_000,
     'risk_tolerance': 3,        'horizon_years': 20,
     'income_stable': True,      'liquidity_need': 'medium',
     'goals': ['retirement', 'property_upgrade'],
     'existing_super_aud': 320_000},
    {'client_id': 'CLI_002', 'name': 'James Okafor',    'age': 58,
     'investable_aud': 2_100_000,'income_aud': 0,
     'risk_tolerance': 2,        'horizon_years': 12,
     'income_stable': False,     'liquidity_need': 'high',
     'goals': ['retirement_income', 'estate_planning'],
     'existing_super_aud': 890_000},
    {'client_id': 'CLI_003', 'name': 'Priya Sharma',   'age': 33,
     'investable_aud': 120_000,  'income_aud': 95_000,
     'risk_tolerance': 4,        'horizon_years': 30,
     'income_stable': True,      'liquidity_need': 'low',
     'goals': ['retirement', 'childrens_education'],
     'existing_super_aud': 68_000},
    {'client_id': 'CLI_004', 'name': 'David Nguyen',   'age': 67,
     'investable_aud': 480_000,  'income_aud': 0,
     'risk_tolerance': 1,        'horizon_years': 8,
     'income_stable': False,     'liquidity_need': 'high',
     'goals': ['capital_preservation', 'income'],
     'existing_super_aud': 210_000},
    {'client_id': 'CLI_005', 'name': 'Emma Russo',     'age': 38,
     'investable_aud': 350_000,  'income_aud': 140_000,
     'risk_tolerance': 5,        'horizon_years': 25,
     'income_stable': True,      'liquidity_need': 'low',
     'goals': ['retirement', 'property_investment'],
     'existing_super_aud': 155_000},
]

# ── Product shelf (investment committee approved — 2025-Q3) ───────────────
PRODUCTS = [
    {'product_id': 'ETF_AU_01', 'name': 'iShares S&P/ASX 200 ETF',
     'type': 'Equity ETF', 'risk_rating': 5, 'min_horizon': 7,
     'fee_pa': 0.07,  'exp_return': 0.085, 'volatility': 0.18,
     'liquidity': 'high',   'esg_score': 0.71, 'trailing_commission': False},
    {'product_id': 'ETF_INTL_01', 'name': 'Vanguard MSCI Index International Shares ETF',
     'type': 'Equity ETF', 'risk_rating': 5, 'min_horizon': 7,
     'fee_pa': 0.18,  'exp_return': 0.090, 'volatility': 0.19,
     'liquidity': 'high',   'esg_score': 0.68, 'trailing_commission': False},
    {'product_id': 'BAL_01', 'name': 'Vanguard Balanced Index Fund',
     'type': 'Balanced Fund', 'risk_rating': 3, 'min_horizon': 5,
     'fee_pa': 0.29,  'exp_return': 0.065, 'volatility': 0.10,
     'liquidity': 'medium',  'esg_score': 0.65, 'trailing_commission': False},
    {'product_id': 'BOND_01', 'name': 'PIMCO Australian Bond Fund',
     'type': 'Fixed Income', 'risk_rating': 2, 'min_horizon': 3,
     'fee_pa': 0.55,  'exp_return': 0.042, 'volatility': 0.05,
     'liquidity': 'medium',  'esg_score': 0.60, 'trailing_commission': False},
    {'product_id': 'SUPER_GROWTH_01', 'name': 'AustralianSuper Growth Option',
     'type': 'Super Option', 'risk_rating': 4, 'min_horizon': 10,
     'fee_pa': 0.67,  'exp_return': 0.078, 'volatility': 0.14,
     'liquidity': 'low',    'esg_score': 0.74, 'trailing_commission': False},
    {'product_id': 'SUPER_BAL_01', 'name': 'AustralianSuper Balanced Option',
     'type': 'Super Option', 'risk_rating': 3, 'min_horizon': 7,
     'fee_pa': 0.56,  'exp_return': 0.063, 'volatility': 0.09,
     'liquidity': 'low',    'esg_score': 0.72, 'trailing_commission': False},
    {'product_id': 'TD_01', 'name': 'CBA 12-Month Term Deposit',
     'type': 'Term Deposit', 'risk_rating': 1, 'min_horizon': 1,
     'fee_pa': 0.00,  'exp_return': 0.047, 'volatility': 0.00,
     'liquidity': 'low',    'esg_score': 0.50, 'trailing_commission': False},
    {'product_id': 'PROP_01', 'name': 'Charter Hall Direct Office Fund',
     'type': 'Property Fund', 'risk_rating': 4, 'min_horizon': 5,
     'fee_pa': 0.90,  'exp_return': 0.072, 'volatility': 0.12,
     'liquidity': 'low',    'esg_score': 0.62, 'trailing_commission': True},
]

clients_df  = pd.DataFrame(CLIENTS)
products_df = pd.DataFrame(PRODUCTS)

print(f'Clients loaded: {len(clients_df)}')
print(f'Products on shelf: {len(products_df)}')
print()
print('Client summary:')
print(clients_df[['client_id','name','age','investable_aud','risk_tolerance','horizon_years']].to_string(index=False))
print()
print('Product shelf (investment committee approved 2025-Q3):')
print(products_df[['product_id','name','type','risk_rating','fee_pa','liquidity']].to_string(index=False))


## Station 2: Feature Engineering — Client Risk Profiling (Eq 9.6)

The composite client risk score aggregates questionnaire responses across four dimensions into a single continuous measure that determines the client's risk category and eligible product universe.

$$\text{CRS}_j = \sum_{i=1}^{m} v_i \cdot q_{i,j} \tag{9.6}$$


In [ ]:
# ── Station 2: Client Risk Profiling — Eq 9.6 ─────────────────────────────

# Questionnaire item weights (v_i, must sum to 1.0)
V_WEIGHTS = {
    'risk_tolerance': 0.35,   # 1–5 scale → normalised to [0,1]
    'horizon_norm':   0.25,   # years normalised by 30-year maximum
    'income_stable':  0.20,   # boolean → 0 or 1
    'liquidity_low':  0.20    # low need = high risk capacity; high need = low risk capacity
}

LIQUIDITY_SCORES = {'low': 1.0, 'medium': 0.5, 'high': 0.0}  # low need → high score

RISK_CATEGORIES = [
    (0.00, 0.25, 'Conservative'),
    (0.25, 0.45, 'Moderate-Conservative'),
    (0.45, 0.65, 'Balanced'),
    (0.65, 0.80, 'Growth'),
    (0.80, 1.01, 'Aggressive Growth'),
]

# IPS target allocations by risk category (investment committee, 2025-Q3)
IPS_ALLOCATIONS = {
    'Conservative':          {'AU_Equities': 0.10, 'Intl_Equities': 0.05, 'Fixed_Income': 0.55, 'Cash': 0.25, 'Alternatives': 0.05},
    'Moderate-Conservative': {'AU_Equities': 0.20, 'Intl_Equities': 0.15, 'Fixed_Income': 0.40, 'Cash': 0.15, 'Alternatives': 0.10},
    'Balanced':              {'AU_Equities': 0.30, 'Intl_Equities': 0.25, 'Fixed_Income': 0.28, 'Cash': 0.07, 'Alternatives': 0.10},
    'Growth':                {'AU_Equities': 0.38, 'Intl_Equities': 0.32, 'Fixed_Income': 0.15, 'Cash': 0.05, 'Alternatives': 0.10},
    'Aggressive Growth':     {'AU_Equities': 0.42, 'Intl_Equities': 0.38, 'Fixed_Income': 0.08, 'Cash': 0.02, 'Alternatives': 0.10},
}

def assign_risk_category(crs):
    for lo, hi, cat in RISK_CATEGORIES:
        if lo <= crs < hi:
            return cat
    return 'Aggressive Growth'

def compute_crs(client):
    q = {
        'risk_tolerance': (client['risk_tolerance'] - 1) / 4,
        'horizon_norm':   min(client['horizon_years'] / 30.0, 1.0),
        'income_stable':  float(client['income_stable']),
        'liquidity_low':  LIQUIDITY_SCORES[client['liquidity_need']]
    }
    crs = sum(V_WEIGHTS[k] * q[k] for k in V_WEIGHTS)
    return round(crs, 4)

# Compute CRS for all clients
profiles = []
for client in CLIENTS:
    crs = compute_crs(client)
    cat = assign_risk_category(crs)
    alloc = IPS_ALLOCATIONS[cat]
    profiles.append({
        'client_id':     client['client_id'],
        'name':          client['name'],
        'crs':           crs,
        'risk_category': cat,
        'allocation':    alloc
    })

profiles_df = pd.DataFrame([{k: v for k, v in p.items() if k != 'allocation'} for p in profiles])

print('Client Risk Profiles (Eq 9.6):')
print(profiles_df.to_string(index=False))
print()
print('Portfolio allocations by risk category (IPS target weights):')
for p in profiles:
    alloc_str = ', '.join(f'{k}: {v*100:.0f}%' for k, v in p['allocation'].items())
    print(f'  {p["name"]:20s} ({p["risk_category"]:22s}): {alloc_str}')


## Station 2–3: Product Suitability Scoring and Recommendation (Eq 9.7)

Each product is scored against each client's profile across four compatibility dimensions. $C_{\text{horizon}}$ is a hard constraint — any product with a minimum recommended horizon exceeding the client's stated horizon receives a score of zero regardless of other dimensions.

$$\text{PSS}_{j,p} = \alpha_1 \cdot C_{\text{risk}}(j,p) + \alpha_2 \cdot C_{\text{horizon}}(j,p) + \alpha_3 \cdot C_{\text{liquidity}}(j,p) + \alpha_4 \cdot C_{\text{fee}}(p) \tag{9.7}$$


In [ ]:
# ── Station 2–3: Product Suitability Scoring — Eq 9.7 ─────────────────────

# Alpha weights (adviser-configurable; investment committee default)
ALPHA = {'risk': 0.40, 'horizon': 0.30, 'liquidity': 0.20, 'fee': 0.10}

PROD_LIQUIDITY_SCORES = {'high': 1.0, 'medium': 0.6, 'low': 0.2}
CLIENT_LIQUIDITY_NEED  = {'high': 0.8, 'medium': 0.5, 'low': 0.2}

def compute_pss(client, product, crs):
    # C_risk: penalty for mismatch between client risk capacity and product risk rating
    client_rating = 1 + crs * 4           # map CRS [0,1] → [1,5]
    c_risk = 1 - abs(client_rating - product['risk_rating']) / 4
    c_risk = max(c_risk, 0.0)

    # C_horizon: HARD CONSTRAINT — zero if product minimum horizon exceeds client horizon
    c_horizon = 1.0 if client['horizon_years'] >= product['min_horizon'] else 0.0

    # C_liquidity: penalise if product liquidity is lower than client need
    prod_liq   = PROD_LIQUIDITY_SCORES[product['liquidity']]
    client_liq = CLIENT_LIQUIDITY_NEED[client['liquidity_need']]
    c_liquidity = prod_liq if prod_liq >= client_liq else prod_liq / client_liq

    # C_fee: fee competitiveness (0% = 1.0; 2.0% p.a. = 0.0)
    c_fee = max(1 - product['fee_pa'] / 2.0, 0.0)

    pss = (ALPHA['risk']      * c_risk +
           ALPHA['horizon']   * c_horizon +
           ALPHA['liquidity'] * c_liquidity +
           ALPHA['fee']       * c_fee)

    return round(pss, 4), {
        'c_risk': round(c_risk, 3),
        'c_horizon': c_horizon,
        'c_liquidity': round(c_liquidity, 3),
        'c_fee': round(c_fee, 3)
    }

# Build full suitability matrix (clients × products)
BID_SUITABILITY_FLOOR = 0.50  # minimum PSS for a product to be recommended
N_RECOMMEND = 3               # number of products to recommend per client

all_scores = []
recommendations = {}          # client_id → list of top-N products with PSS + components

for profile in profiles:
    client = next(c for c in CLIENTS if c['client_id'] == profile['client_id'])
    crs    = profile['crs']
    client_recs = []
    for product in PRODUCTS:
        pss, components = compute_pss(client, product, crs)
        all_scores.append({
            'client_id':  profile['client_id'],
            'client':     profile['name'],
            'product_id': product['product_id'],
            'product':    product['name'],
            'pss':        pss,
            **components
        })
        if pss >= BID_SUITABILITY_FLOOR:
            client_recs.append({'product': product, 'pss': pss, 'components': components})
    client_recs.sort(key=lambda x: x['pss'], reverse=True)
    recommendations[profile['client_id']] = client_recs[:N_RECOMMEND]

scores_df = pd.DataFrame(all_scores)

print('Product Suitability Matrix — Top recommendations per client (Eq 9.7):')
print(f'BID suitability floor: PSS >= {BID_SUITABILITY_FLOOR}')
print()
for profile in profiles:
    cid = profile['client_id']
    recs = recommendations[cid]
    print(f'{profile["name"]} ({profile["risk_category"]}, CRS={profile["crs"]:.3f}):')
    for r in recs:
        c = r['components']
        print(f'  {r["product"]["name"]:45s} PSS={r["pss"]:.3f}  '
              f'(risk={c["c_risk"]:.2f} horizon={c["c_horizon"]:.0f} liq={c["c_liquidity"]:.2f} fee={c["c_fee"]:.2f})')
    print()


## Station 3: Model Design — Best Interests Duty Compliance Gate

Before generating the SOA draft, the platform runs a rule-based BID compliance gate evaluating the advice session against the six-step test specified in ASIC Regulatory Guide 175. Any session failing a hard check is blocked from SOA generation and escalated to the Responsible Manager.


In [ ]:
# ── Station 3: Best Interests Duty Compliance Gate ─────────────────────────

def bid_compliance_gate(client, profile, recs):
    checks = {}
    rec_products = [r['product'] for r in recs]
    crs = profile['crs']

    # Check 1 (HARD): Risk profile documented — questionnaire completed
    checks['risk_profile_documented'] = (
        True,           # always True if we reach this point
        'Questionnaire completed — risk profile on file.',
        'HARD'
    )

    # Check 2 (HARD): Horizon compatibility — no product with c_horizon=0
    horizon_ok = all(r['components']['c_horizon'] == 1.0 for r in recs)
    checks['horizon_compatibility'] = (
        horizon_ok,
        'All recommended products within client investment horizon.' if horizon_ok
            else 'FAIL: product minimum horizon exceeds client horizon.',
        'HARD'
    )

    # Check 3 (SOFT): Fee reasonableness — portfolio average fee < 1.5% p.a.
    avg_fee = float(np.mean([p['fee_pa'] for p in rec_products]))
    fee_ok  = avg_fee < 1.50
    checks['fees_reasonable'] = (
        fee_ok,
        f'Average portfolio fee: {avg_fee:.2f}% p.a. (< 1.5% threshold).' if fee_ok
            else f'FLAG: average fee {avg_fee:.2f}% p.a. exceeds 1.5% — adviser review required.',
        'SOFT'
    )

    # Check 4 (SOFT): Risk alignment — all products within 1.5 rating points of client rating
    client_rating = 1 + crs * 4
    risk_ok = all(abs(p['risk_rating'] - client_rating) <= 1.5 for p in rec_products)
    checks['risk_alignment'] = (
        risk_ok,
        'All products within acceptable risk band of client profile.' if risk_ok
            else 'FLAG: one or more products outside acceptable risk band.',
        'SOFT'
    )

    # Check 5 (SOFT): Diversification — at least 2 different product types
    product_types = set(p['type'] for p in rec_products)
    div_ok = len(product_types) >= 2
    checks['diversification'] = (
        div_ok,
        f'Product types recommended: {list(product_types)}.' if div_ok
            else 'FLAG: all products in same category — insufficient diversification.',
        'SOFT'
    )

    # Check 6 (HARD): Conflict disclosure — flag any products with trailing commissions
    trailing = [p['name'] for p in rec_products if p.get('trailing_commission')]
    conflict_ok = True  # always passes — we disclose rather than block
    checks['conflict_disclosure'] = (
        True,
        f'Trailing commissions present on: {trailing} — auto-disclosed in SOA.' if trailing
            else 'No trailing commissions — no conflict disclosure required.',
        'SOFT'
    )

    # Overall gate: HARD checks must all pass for SOA generation to proceed
    hard_passed = all(v[0] for k, v in checks.items() if v[2] == 'HARD')
    soft_flags  = [k for k, v in checks.items() if v[2] == 'SOFT' and not v[0]]
    return hard_passed, soft_flags, checks

print('Best Interests Duty Compliance Gate — ASIC RG 175')
print('=' * 70)
bid_results = {}
for profile in profiles:
    cid    = profile['client_id']
    client = next(c for c in CLIENTS if c['client_id'] == cid)
    recs   = recommendations[cid]
    if not recs:
        print(f'{profile["name"]}: NO SUITABLE PRODUCTS — escalate to RM')
        continue
    passed, soft_flags, checks = bid_compliance_gate(client, profile, recs)
    bid_results[cid] = {'passed': passed, 'soft_flags': soft_flags, 'checks': checks}
    status = 'PASS — SOA generation authorised' if passed else 'BLOCK — escalate to RM'
    print(f'{profile["name"]:20s}: {status}')
    for check_name, (ok, msg, severity) in checks.items():
        icon = '[OK]' if ok else '[FLAG]'
        print(f'  {icon} {severity:4s} {check_name:30s}: {msg}')
    print()


## Station 4: Implementation — SOA Draft Generation (Gemini / Mock)

For each client whose advice session passes the BID gate, Gemini 1.5 Flash generates a first-draft Statement of Advice narrative. The mock path produces a structurally identical draft when no API key is available. Both paths feed the same downstream audit trail and adviser review workflow.


In [ ]:
# ── Station 4: SOA Draft Generation (Gemini / Mock) ───────────────────────

AUDIT_TRAIL = []
SOA_TEMPLATE_VERSION  = 'soa-v4'
RECOMMENDER_VERSION   = 'wealthtech_advisor_recommender_v2.1'
RECOMMENDER_HASH      = hashlib.md5(b'PSS-v2-alpha-0.40-0.30-0.20-0.10').hexdigest()[:12]

def build_soa_prompt(client, profile, recs, alloc):
    prod_lines  = '\n'.join(
        f'- {r["product"]["name"]} ({r["product"]["type"]}, '
        f'risk rating {r["product"]["risk_rating"]}/5, '
        f'fee {r["product"]["fee_pa"]:.2f}% p.a., '
        f'expected return {r["product"]["exp_return"]*100:.1f}% p.a.): '
        f'suitability score {r["pss"]:.3f}'
        for r in recs
    )
    alloc_lines = '\n'.join(f'- {k}: {v*100:.0f}%' for k, v in alloc.items())
    goals_str   = ', '.join(client['goals'])
    return (
        'You are a senior financial adviser at an Australian wealth management firm holding an AFSL. '
        'Generate a professional Statement of Advice (SOA) draft compliant with ASIC Regulatory Guide 175.\n\n'
        f'CLIENT: {client["name"]} | Age: {client["age"]} | '
        f'Investable assets: AUD {client["investable_aud"]:,.0f} | '
        f'Annual income: AUD {client["income_aud"]:,.0f}\n'
        f'Goals: {goals_str}\n'
        f'Risk profile: {profile["risk_category"]} (CRS = {profile["crs"]:.3f})\n'
        f'Investment horizon: {client["horizon_years"]} years\n\n'
        f'Recommended products (ranked by suitability score Eq 9.7):\n{prod_lines}\n\n'
        f'Proposed strategic asset allocation (IPS target weights):\n{alloc_lines}\n\n'
        'Write a professional SOA with these sections:\n'
        '1. Scope of Advice and Basis for Engagement\n'
        '2. Your Current Situation\n'
        '3. Your Goals and Objectives\n'
        '4. Our Recommendations — Basis for Advice (explain why each product satisfies '
        'Best Interests Duty s.961B Corporations Act 2001)\n'
        '5. Proposed Portfolio and Expected Outcomes\n'
        '6. Risks and Key Considerations\n'
        '7. Fees and Charges\n'
        'Conclude with: [DRAFT — PENDING ADVISER REVIEW AND RESPONSIBLE MANAGER SIGN-OFF]'
    )

def mock_soa(client, profile, recs, alloc):
    prod_text  = '; '.join(
        f'{r["product"]["name"]} (PSS {r["pss"]:.3f}, fee {r["product"]["fee_pa"]:.2f}% p.a.)'
        for r in recs
    )
    alloc_text = ', '.join(f'{k} {v*100:.0f}%' for k, v in alloc.items())
    exp_return = float(np.mean([r['product']['exp_return'] for r in recs])) * 100
    return (
        f'STATEMENT OF ADVICE — DRAFT\n'
        f'Client: {client["name"]} | Prepared by: [Adviser Name, AFSL 000000]\n'
        f'Date: {datetime.date.today()}\n\n'
        f'1. SCOPE OF ADVICE\n'
        f'This Statement of Advice covers investment strategy, product selection, and portfolio '
        f'construction for {client["name"]}. The advice addresses the following goals: '
        f'{", ".join(client["goals"])}. Advice is limited to the financial products listed below.\n\n'
        f'2. YOUR CURRENT SITUATION\n'
        f'{client["name"]} is aged {client["age"]} with AUD {client["investable_aud"]:,.0f} '
        f'available to invest, AUD {client["existing_super_aud"]:,.0f} in existing superannuation, '
        f'and an investment horizon of {client["horizon_years"]} years. '
        f'Risk profile: {profile["risk_category"]} (composite risk score {profile["crs"]:.3f}, Eq 9.6).\n\n'
        f'3. YOUR GOALS\n'
        f'Primary goals: {", ".join(client["goals"])}. '
        f'These goals have been assessed as achievable within the proposed strategy given the '
        f'client\'s current asset base and investment horizon.\n\n'
        f'4. OUR RECOMMENDATIONS\n'
        f'The following products are recommended based on suitability scoring (Eq 9.7) and '
        f'the Best Interests Duty test (ASIC RG 175, Corporations Act s.961B):\n'
        f'{prod_text}\n'
        f'Each product satisfies the horizon compatibility constraint, falls within the '
        f'client\'s risk band ({profile["risk_category"]}), and has been assessed as '
        f'likely to leave the client in a better financial position than the status quo.\n\n'
        f'5. PROPOSED PORTFOLIO\n'
        f'Strategic asset allocation: {alloc_text}.\n'
        f'Expected portfolio return: approximately {exp_return:.1f}% p.a. (gross of fees, '
        f'based on long-run asset class assumptions from the Investment Policy Statement).\n\n'
        f'6. RISKS\n'
        f'Investments carry market risk, sequence-of-returns risk, and currency risk (for '
        f'international allocations). Past performance is not indicative of future results. '
        f'The proposed portfolio may experience periods of negative returns, particularly '
        f'during equity market downturns, which may impact ability to meet short-term goals.\n\n'
        f'7. FEES AND CHARGES\n'
        f'Weighted average product fee: {float(np.mean([r["product"]["fee_pa"] for r in recs])):.2f}% p.a. '
        f'Adviser fee: [to be disclosed separately under FSG]. '
        f'No trailing commissions apply to the recommended products [except as separately disclosed].\n\n'
        f'[DRAFT — PENDING ADVISER REVIEW AND RESPONSIBLE MANAGER SIGN-OFF]'
    )

def generate_soa(client, profile, recs):
    alloc = profile['allocation']
    if USE_LIVE_API:
        model_obj = genai.GenerativeModel(GEMINI_MODEL)
        response  = model_obj.generate_content(build_soa_prompt(client, profile, recs, alloc))
        soa_text  = response.text
        source    = 'gemini-live'
    else:
        soa_text  = mock_soa(client, profile, recs, alloc)
        source    = 'mock'

    entry = {
        'session_id':           f'ADV-{datetime.date.today()}-{client["client_id"]}',
        'client_id':            client['client_id'],
        'client_name':          client['name'],
        'crs':                  profile['crs'],
        'risk_category':        profile['risk_category'],
        'products_recommended': [r['product']['product_id'] for r in recs],
        'pss_scores':           [r['pss'] for r in recs],
        'bid_gate_passed':      True,
        'llm_model':            GEMINI_MODEL if USE_LIVE_API else 'mock',
        'llm_source':           source,
        'soa_template':         SOA_TEMPLATE_VERSION,
        'recommender_version':  RECOMMENDER_VERSION,
        'recommender_hash':     RECOMMENDER_HASH,
        'generated_ts':         datetime.datetime.now().isoformat(),
        'adviser_id':           None,
        'rm_approval_id':       None,
        'soa_delivered_ts':     None,
        'status':               'PENDING_ADVISER_REVIEW'
    }
    AUDIT_TRAIL.append(entry)
    return soa_text, entry

# Generate SOAs for all clients that passed the BID gate
soa_outputs = {}
for profile in profiles:
    cid    = profile['client_id']
    client = next(c for c in CLIENTS if c['client_id'] == cid)
    recs   = recommendations.get(cid, [])
    bid    = bid_results.get(cid)
    if not bid or not bid['passed'] or not recs:
        print(f'{client["name"]}: BID gate failed or no recommendations — SOA not generated.')
        continue
    soa_text, audit_entry = generate_soa(client, profile, recs)
    soa_outputs[cid] = soa_text
    print(f'SOA generated for {client["name"]} ({profile["risk_category"]}): {len(soa_text)} chars [{audit_entry["llm_source"]}]')

print()
# Display first SOA draft
first_cid  = profiles[0]['client_id']
first_name = profiles[0]['name']
print(f'=== SOA DRAFT: {first_name} ===')
print(soa_outputs.get(first_cid, 'No SOA generated.'))


## Station 4: Implementation — Adviser Dashboard Visualisations

The Streamlit adviser workstation renders four panels: client risk profile distribution, product suitability heatmap, proposed portfolio allocations, and model governance audit summary. Static matplotlib figures are shown here.


In [ ]:
# ── Station 4: Adviser Dashboard Visualisations ────────────────────────────

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('WealthTech Financial Advisory Platform — Adviser Dashboard',
              fontsize=15, fontweight='bold', y=0.99)

CATEGORY_COLOURS = {
    'Conservative': '#4472C4',
    'Moderate-Conservative': '#70AD47',
    'Balanced': '#FFC000',
    'Growth': '#ED7D31',
    'Aggressive Growth': '#C00000'
}

# Panel 1: Client risk scores and categories
ax = axes[0, 0]
names  = [p['name'].split()[0] for p in profiles]
crss   = [p['crs'] for p in profiles]
cats   = [p['risk_category'] for p in profiles]
colors = [CATEGORY_COLOURS[c] for c in cats]
bars   = ax.barh(names, crss, color=colors, alpha=0.85)
for lo, hi, cat in RISK_CATEGORIES:
    ax.axvspan(lo, hi, alpha=0.06, color=CATEGORY_COLOURS[cat])
for bar, crs, cat in zip(bars, crss, cats):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height() / 2,
             f'{crs:.3f} ({cat})', va='center', fontsize=9)
ax.set_xlabel('Client Risk Score — CRS (Eq 9.6)', fontsize=11)
ax.set_title('Client Risk Profiles', fontsize=12, fontweight='bold')
ax.set_xlim(0, 1.05)
ax.grid(alpha=0.3, axis='x')

# Panel 2: Product suitability heatmap (PSS matrix)
ax = axes[0, 1]
pivot = scores_df.pivot(index='client', columns='product', values='pss')
pivot.columns = [c[:18] for c in pivot.columns]
im = ax.imshow(pivot.values, cmap='RdYlGn', vmin=0, vmax=1, aspect='auto')
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns, rotation=35, ha='right', fontsize=8)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels([n.split()[0] for n in pivot.index], fontsize=9)
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        val = pivot.values[i, j]
        ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=7.5,
                 color='white' if val < 0.25 else 'black')
ax.set_title('Product Suitability Matrix — PSS (Eq 9.7)', fontsize=12, fontweight='bold')
plt.colorbar(im, ax=ax, shrink=0.8, label='Suitability score')

# Panel 3: Portfolio allocations by client
ax = axes[1, 0]
asset_classes = list(IPS_ALLOCATIONS['Balanced'].keys())
asset_colors  = ['#4472C4', '#ED7D31', '#70AD47', '#FFC000', '#9E480E']
x = np.arange(len(profiles))
bottom = np.zeros(len(profiles))
for ac, col in zip(asset_classes, asset_colors):
    weights = [IPS_ALLOCATIONS[p['risk_category']][ac] for p in profiles]
    ax.bar(x, weights, bottom=bottom, label=ac, color=col, alpha=0.85)
    bottom += np.array(weights)
ax.set_xticks(x)
ax.set_xticklabels([p['name'].split()[0] for p in profiles], fontsize=10)
ax.set_ylabel('Portfolio weight', fontsize=11)
ax.set_title('Proposed Strategic Asset Allocation (IPS Target Weights)', fontsize=12, fontweight='bold')
ax.legend(fontsize=8, loc='upper left')
ax.set_ylim(0, 1.05)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
ax.grid(alpha=0.3, axis='y')

# Panel 4: Model governance — PSS score distribution + audit summary
ax = axes[1, 1]
all_pss = scores_df['pss'].values
rec_pss = [r['pss'] for cid in recommendations for r in recommendations[cid]]
ax.hist(all_pss, bins=25, alpha=0.6, color='steelblue', label='All client-product pairs')
ax.hist(rec_pss, bins=15, alpha=0.85, color='#70AD47', label='Recommended products')
ax.axvline(BID_SUITABILITY_FLOOR, color='crimson', ls='--', lw=2, label=f'BID floor (PSS={BID_SUITABILITY_FLOOR})')
ax.set_xlabel('Suitability Score PSS (Eq 9.7)', fontsize=11)
ax.set_ylabel('Count', fontsize=11)
ax.set_title('PSS Distribution — Recommended vs. Full Product Shelf', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('week9_advisory_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print('Dashboard saved: week9_advisory_dashboard.png')

## Station 4: Model Governance — Audit Trail and SR 11-7 Summary

The advice session audit trail captures all model versions, input scores, BID gate outcomes, LLM provenance, and approval workflow actions. Under ASIC RG 255 and SR 11-7, this constitutes the evidential record that the advice process satisfied the Best Interests Duty and that model use governance was maintained.


In [ ]:
# ── Station 4: Model Governance Audit Trail ────────────────────────────────

# Simulate adviser review and RM approval for first session
if AUDIT_TRAIL:
    AUDIT_TRAIL[0]['adviser_id']      = 'ADV_007'
    AUDIT_TRAIL[0]['rm_approval_id']  = 'RM_003'
    AUDIT_TRAIL[0]['soa_delivered_ts'] = datetime.datetime.now().isoformat()
    AUDIT_TRAIL[0]['status']          = 'SOA_DELIVERED'

print('=' * 65)
print('WEALTHTECH ADVISORY PLATFORM — MODEL GOVERNANCE SUMMARY')
print('=' * 65)
print()
print('Recommendation Engine:')
print(f'  Model:            {RECOMMENDER_VERSION}')
print(f'  Scoring formula:  PSS-v2 (Eq 9.7, α = 0.40 / 0.30 / 0.20 / 0.10)')
print(f'  Formula hash:     {RECOMMENDER_HASH}')
print(f'  Product shelf:    2025-Q3 (8 products, investment committee approved)')
print(f'  BID floor:        PSS >= {BID_SUITABILITY_FLOOR}')
print(f'  Validation:       Approved 2025-09-01 — peer review by Head of Advice')
print(f'  Monitor trigger:  Adviser override rate > 20% per product category')
print(f'  Next review:      2025-12-01 (quarterly IPS cycle)')
print()
print('SOA Generation:')
print(f'  LLM:              {GEMINI_MODEL if USE_LIVE_API else "mock"} ({"live" if USE_LIVE_API else "mock"})')
print(f'  Template version: {SOA_TEMPLATE_VERSION}')
print(f'  HITL requirement: adviser review + RM sign-off before delivery (ASIC RG 175)')
print()
print('Advice Sessions This Run:')
print(f'  Sessions generated: {len(AUDIT_TRAIL)}')
print(f'  SOA delivered:      {sum(1 for e in AUDIT_TRAIL if e["status"] == "SOA_DELIVERED")}')
print(f'  Pending review:     {sum(1 for e in AUDIT_TRAIL if e["status"] == "PENDING_ADVISER_REVIEW")}')
print()
audit_df = pd.DataFrame(AUDIT_TRAIL)
disp = ['session_id', 'client_name', 'risk_category', 'crs', 'bid_gate_passed',
        'llm_source', 'adviser_id', 'status']
print('Audit Trail:')
print(audit_df[disp].to_string(index=False))
print()
print('Export: audit_df.to_csv("asic_advice_audit_trail.csv", index=False)')
print('This trail constitutes the ASIC RG 175 / RG 255 evidential record for each advice session.')


## Extension: Streamlit Adviser Workstation

The cell below contains a complete Streamlit application that renders the WealthTech advisory platform as an interactive adviser workstation. Copy the code to `advisory_app.py` and run:

```bash
pip install streamlit scikit-learn matplotlib numpy pandas
streamlit run advisory_app.py
```

The workstation provides four tabs: **Client Profiling** (questionnaire input → CRS and risk category), **Product Recommendations** (PSS scores and component breakdown for top-N products), **Portfolio Allocation** (IPS target-weight allocation chart), and **SOA & Compliance** (SOA draft display, BID gate results, audit trail export).


In [ ]:
STREAMLIT_CODE = '''
# advisory_app.py — WealthTech Financial Advisory Platform (Week 9)
# Run: streamlit run advisory_app.py

import streamlit as st
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

st.set_page_config(page_title="WealthTech Advisory Platform", layout="wide")
st.title("WealthTech Financial Advisory Platform")
st.caption("FINS5557 Week 9 | Hric & Lin, 2026 | For use by licensed financial advisers under AFSL")

with st.sidebar:
    st.header("Adviser Settings")
    adviser_id = st.text_input("Adviser ID", value="ADV_007")
    st.divider()
    st.write("**Model Governance**")
    st.write("Recommender: PSS-v2 (Eq 9.7)")
    st.write("Alpha weights: 0.40 / 0.30 / 0.20 / 0.10")
    st.write("Product shelf: 2025-Q3")
    st.write("BID floor: PSS >= 0.50")
    st.write("Validation: 2025-09-01")
    st.write("SR 11-7 status: Approved")

tab1, tab2, tab3, tab4 = st.tabs([
    "Client Profiling",
    "Product Recommendations",
    "Portfolio Allocation",
    "SOA & Compliance"
])

with tab1:
    st.subheader("Client Risk Profiling — Eq 9.6")
    col1, col2 = st.columns(2)
    with col1:
        client_name   = st.text_input("Client name", value="Sarah Chen")
        age           = st.number_input("Age", min_value=18, max_value=90, value=45)
        investable    = st.number_input("Investable assets (AUD)", value=850000, step=10000)
        income        = st.number_input("Annual income (AUD)", value=180000, step=5000)
    with col2:
        risk_tolerance = st.slider("Risk tolerance (1=Very Conservative, 5=Aggressive)", 1, 5, 3)
        horizon        = st.slider("Investment horizon (years)", 1, 40, 20)
        income_stable  = st.checkbox("Income is stable / employment-based", value=True)
        liquidity_need = st.select_slider("Liquidity need", options=["low", "medium", "high"], value="medium")

    LIQUIDITY_MAP = {"low": 1.0, "medium": 0.5, "high": 0.0}
    V = {"risk_tolerance": 0.35, "horizon_norm": 0.25, "income_stable": 0.20, "liquidity_low": 0.20}
    q = {
        "risk_tolerance": (risk_tolerance - 1) / 4,
        "horizon_norm":   min(horizon / 30.0, 1.0),
        "income_stable":  float(income_stable),
        "liquidity_low":  LIQUIDITY_MAP[liquidity_need]
    }
    crs = sum(V[k] * q[k] for k in V)
    CATS = [(0, 0.25, "Conservative"), (0.25, 0.45, "Moderate-Conservative"),
             (0.45, 0.65, "Balanced"), (0.65, 0.80, "Growth"), (0.80, 1.01, "Aggressive Growth")]
    category = next(cat for lo, hi, cat in CATS if lo <= crs < hi)

    st.metric("Client Risk Score (CRS)", f"{crs:.3f}", help="Eq 9.6")
    st.success(f"Risk Category: **{category}**")

with tab2:
    st.subheader("Product Suitability Scores — Eq 9.7")
    st.info("Connect to: scores_df filtered by client_id from notebook Station 2-3")
    st.write("Products ranked by PSS score. BID floor: PSS >= 0.50 for recommendation.")
    demo_recs = pd.DataFrame({
        "Product":    ["Vanguard Balanced Index Fund", "AustralianSuper Balanced", "iShares S&P/ASX 200 ETF"],
        "PSS":        [0.842, 0.798, 0.751],
        "Risk fit":   [0.92, 0.88, 0.75],
        "Horizon":    [1.00, 1.00, 1.00],
        "Liquidity":  [0.85, 0.60, 0.95],
        "Fee score":  [0.86, 0.72, 0.96],
        "BID gate":   [True, True, True]
    })
    st.dataframe(demo_recs, use_container_width=True)

with tab3:
    st.subheader("Proposed Strategic Asset Allocation (IPS Target Weights)")
    IPS = {
        "Conservative":          {"AU Equities": 0.10, "Intl Equities": 0.05, "Fixed Income": 0.55, "Cash": 0.25, "Alternatives": 0.05},
        "Moderate-Conservative": {"AU Equities": 0.20, "Intl Equities": 0.15, "Fixed Income": 0.40, "Cash": 0.15, "Alternatives": 0.10},
        "Balanced":              {"AU Equities": 0.30, "Intl Equities": 0.25, "Fixed Income": 0.28, "Cash": 0.07, "Alternatives": 0.10},
        "Growth":                {"AU Equities": 0.38, "Intl Equities": 0.32, "Fixed Income": 0.15, "Cash": 0.05, "Alternatives": 0.10},
        "Aggressive Growth":     {"AU Equities": 0.42, "Intl Equities": 0.38, "Fixed Income": 0.08, "Cash": 0.02, "Alternatives": 0.10},
    }
    alloc = IPS.get(category, IPS["Balanced"])
    alloc_df = pd.DataFrame.from_dict(alloc, orient="index", columns=["Weight"])
    alloc_df["Weight %"] = (alloc_df["Weight"] * 100).round(0).astype(int)
    st.bar_chart(alloc_df["Weight"])
    st.dataframe(alloc_df[["Weight %"]], use_container_width=False)

with tab4:
    st.subheader("Statement of Advice — Draft + BID Compliance")
    st.warning("This is an AI-generated draft. The licensed adviser must review, edit, and approve before delivery. The AFSL holder bears responsibility for compliance with Best Interests Duty (Corporations Act s.961B).")
    st.write("Connect to: `soa_outputs[client_id]` from notebook Station 4")
    st.text_area("SOA Draft (adviser editable)", height=300,
                  value="[SOA draft will appear here after generation — connect to notebook pipeline output]")
    st.subheader("BID Compliance Gate Results")
    bid_demo = pd.DataFrame({
        "Check":          ["Risk profile documented", "Horizon compatibility", "Fee reasonableness",
                           "Risk alignment", "Diversification", "Conflict disclosure"],
        "Type":           ["HARD", "HARD", "SOFT", "SOFT", "SOFT", "SOFT"],
        "Result":         [True, True, True, True, True, True],
        "Note":           ["Questionnaire on file", "All products within horizon",
                           "Avg fee 0.44% p.a.", "All in risk band", "3 product types", "No trailing commissions"]
    })
    st.dataframe(bid_demo, use_container_width=True)
    st.success("BID gate: PASSED — SOA generation authorised. RM sign-off required before delivery.")
    if st.button("Export Audit Trail (CSV)"):
        st.info("In production: audit_df.to_csv('asic_advice_audit_trail.csv', index=False)")
'''

with open('advisory_app.py', 'w') as f:
    f.write(STREAMLIT_CODE.strip())

print('Streamlit adviser workstation written to advisory_app.py')
print('Run: streamlit run advisory_app.py')
print()
print('Four tabs:')
print('  Tab 1 — Client Profiling:         questionnaire → CRS (Eq 9.6) + risk category')
print('  Tab 2 — Product Recommendations:  PSS scores (Eq 9.7) + component breakdown')
print('  Tab 3 — Portfolio Allocation:     IPS target-weight bar chart')
print('  Tab 4 — SOA & Compliance:         draft SOA + BID gate results + audit trail export')
